# Lab 1 — A memory store from scratch (EP12)

The theory video opened with a research assistant that remembered nothing of Monday by Tuesday.
In this lab we see that problem for real, then write a **memory store** after the book's Listing 17.1 (§17.4)

**What you'll do**
1. Talk on Monday, then ask on Tuesday **without memory**
2. `MemoryEntry` — one memory: text, embedding, time, importance
3. `VectorMemoryStore` — **write** (skip duplicates) · **retrieve** (similarity + recency) · **evict** (when full)
4. Tuesday **with memory** — and why the first try still misses

The memories' times are simulated ("yesterday", "two years ago"); every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §17.1, §17.4, §17.5.2, Listing 17.1 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep12-lab1-memory-store.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs less than 0.1 US cents) |
| Packages | `openai`, `python-dotenv`, `numpy` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv numpy

## 1. Two models: one answers, one embeds

**The answering model `google/gemma-4-26b-a4b-it`** — the same model as every episode, pinned to Makora.
USD 0.09 per million input tokens and USD 0.28 per million output tokens
· The `:free` version doesn't work (tested 2026-11-22: 3 out of 3 calls got 429 "rate-limited")

**The embedding model `qwen/qwen3-embedding-8b`** — the same as EP10–EP11, pinned to Nebius
— every memory becomes a vector, so we can search by meaning

Every request's real cost is added up in `COST`

In [ ]:
import os
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import numpy as np
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}

In [ ]:
KEY = load_key()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=KEY, max_retries=5)
COST = {"chat": 0.0, "embed": 0.0}

def chat(messages):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=messages)
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

def embed(text):
    r = client.embeddings.create(model=EMBED_MODEL, input=[text], extra_body=EMBED_EXTRA)
    COST["embed"] += r.usage.cost
    v = np.array(r.data[0].embedding)
    return v / np.linalg.norm(v)                         # length 1: dot product = cosine

## 2. Monday and Tuesday — no memory

An LLM is stateless (§17.1): on every call it sees only the messages we send in that call.
Tuesday is a **new session** — the message list starts empty

In [ ]:
monday = [{"role": "user", "content": "I'm writing a report on transformers for time series. "
                                      "Please keep your answers short."}]
print(chat(monday))

In [ ]:
tuesday = [{"role": "user", "content": "Any new papers for my report?"}]
print(chat(tuesday)[:600])

## 3. One memory: `MemoryEntry`

After Listing 17.1: the text, its embedding, the **time** it was written, its **importance** (0–1), how often it was retrieved, and its source

In [ ]:
@dataclass
class MemoryEntry:
    content: str
    embedding: np.ndarray
    timestamp: datetime = field(default_factory=datetime.now)
    importance: float = 0.5
    access_count: int = 0
    last_accessed: datetime | None = None
    source: str = "conversation"

## 4. The memory store: `VectorMemoryStore`

| method | what it does |
|---|---|
| `similarities(q)` | the cosine of vector `q` with every memory |
| `recency(entry, now)` | freshness `exp(−decay × hours)` — 1 means just written |
| `write(...)` | skip if a duplicate (cosine > 0.95) · if full → `evict()` · then store |
| `evict()` | drop the one with the lowest `importance × 0.7 + recency × 0.3` |
| `retrieve(q, k)` | score `0.7 × sim + 0.3 × recency` → the top k |

The book's `decay_rate` is 0.01 per hour (halves in ~3 days) — our memories span years, so we use 0.001 (halves in ~29 days)

In [ ]:
class VectorMemoryStore:
    def __init__(self, embed_fn, max_entries=10_000, decay_rate=0.001, recency_weight=0.3):
        self.embed_fn = embed_fn            # text -> unit vector
        self.max_entries = max_entries
        self.decay_rate = decay_rate        # per hour (the book uses 0.01)
        self.recency_weight = recency_weight
        self.entries = []

    # ✏️ TODO 1 (▶ video 3:57): stack every memory's vector into a matrix and multiply by q — every cosine at once

    def similarities(self, q):
        return ___

    # ✏️ TODO 2 (▶ video 4:12): compute the age in hours and return exp(−decay_rate × hours)

    def recency(self, entry, now):
        hours = ___
        return ___

    def write(self, content, importance=0.5, when=None, check_duplicates=True):
        emb = self.embed_fn(content)
        closest = self.similarities(emb).max() if self.entries else 0.0
        if check_duplicates and closest > 0.95:
            print(f"skip   (closest {closest:.2f}): {content}")
            return None
        if len(self.entries) >= self.max_entries:
            self.evict()
        entry = MemoryEntry(content, emb, when or datetime.now(), importance)
        self.entries.append(entry)
        print(f"stored (closest {closest:.2f}): {content}")
        return entry

    def evict(self):
        now, w = datetime.now(), self.recency_weight
        scores = [e.importance * (1 - w) + self.recency(e, now) * w for e in self.entries]
        # ✏️ TODO 3 (▶ video 5:14): remove the lowest-scoring memory from self.entries
        gone = ___
        print(f"evict  (score {min(scores):.2f}): {gone.content}")

    # ✏️ TODO 4 (▶ video 5:19): combined score = (1 − w) × similarity + w × recency (or similarity alone if recency_boost is off)

    def retrieve(self, query, k=5, recency_boost=True):
        sims = self.similarities(self.embed_fn(query))
        now, w = datetime.now(), self.recency_weight
        scores = [___ if recency_boost else s
                  for e, s in zip(self.entries, sims)]
        top = list(np.argsort(scores)[::-1][:k])
        for i in top:
            self.entries[i].access_count += 1
            self.entries[i].last_accessed = now
        return [(scores[i], sims[i], self.entries[i]) for i in top]

In [ ]:
def show(hits):
    for score, sim, e in hits:
        age = (datetime.now() - e.timestamp).days
        print(f"{score:.2f}  sim {sim:.2f}  {age:>3} days  {e.content}")

## 5. Writing Monday's memories

A small store (`max_entries=6`), so we can watch it evict · The times are simulated: **yesterday**
We set the importance ourselves for now — in Lab 2 the model scores it

In [ ]:
mem = VectorMemoryStore(embed, max_entries=6)
yesterday = datetime.now() - timedelta(days=1)               # simulated: Monday
mem.write("The user is writing a report on transformers for time series.", importance=0.9, when=yesterday)
mem.write("The user prefers short answers.", importance=0.8, when=yesterday)
mem.write("The user said thanks.", importance=0.1, when=yesterday)
mem.write("The user prefers short answers.", importance=0.8, when=yesterday)
mem.write("The user wants answers kept brief.", importance=0.8, when=yesterday)
print(len(mem.entries), "memories")

## 6. Retrieval by similarity + recency

Two memories about the editor: **vim** (two years ago) and **VS Code** (last week) — an old fact that has since been replaced

In [ ]:
mem.write("The user's code editor is vim.", importance=0.6,
          when=datetime.now() - timedelta(days=730))
mem.write("The user switched their code editor to VS Code.", importance=0.6,
          when=datetime.now() - timedelta(days=7))
show(mem.retrieve("Which code editor do I use?", k=2, recency_boost=False))
print()
show(mem.retrieve("Which code editor do I use?", k=2))

## 7. A full store: drop what is unimportant and old

There are 6 memories now = `max_entries` · the 7th forces one out

In [ ]:
mem.write("The user's report is due on Friday.", importance=0.8)
print(len(mem.entries), "memories")

## 8. Tuesday — with memory (session continuity, §17.5.2)

Before answering: retrieve the memories related to the question → put them in the system prompt → ask the model

In [ ]:
def tuesday(question, k):
    hits = mem.retrieve(question, k=k)
    show(hits)
    # ✏️ TODO 5 (▶ video 8:14): put the retrieved memories in the system prompt, one per line, then ask the model
    system = "What you remember about this user:\n" + ___
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": question}])

print(tuesday("Any new papers for my report?", k=3))

In [ ]:
print(tuesday("Any new papers for my report?", k=4))

In [ ]:
print({k: f"${v:.5f}" for k, v in COST.items()})

## Summary

| Step | What we saw |
|---|---|
| Tuesday, no memory | asks for the topic + a long answer — forgot both things |
| write: word-for-word copy | skipped (closest 1.00) |
| write: different words, same meaning | stored (closest 0.94 < 0.95) |
| editor question, similarity only | VS Code 0.77 · vim 0.74 — contradicting facts |
| editor question, + recency | VS Code first · vim drops out (but "short answers" takes its place) |
| full store | evicts "said thanks" (0.36) |
| Tuesday, k=3 | the topic is missing — the duplicate takes two slots → still asks for the topic |
| Tuesday, k=4 | a short answer about transformers for time series |

**Exercises**
1. Set `decay_rate=0.01` (the book's value) and ask about the editor again — what comes back, and why?
2. Lower the duplicate threshold to 0.90 — is "wants answers kept brief" skipped? Are the vim and VS Code memories skipped too?
3. Write `get_stats()` as in the book: the number of memories, their average importance, and the oldest memory

**Next — Lab 2:** the model decides what's worth remembering, extracts it as short facts, merges duplicates, handles memories that contradict, and reflects them into lessons